# Analysis of data for the Essential FFPE Panel

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score
from sklearn.neighbors import NearestNeighbors
from scipy.stats import gaussian_kde
from sklearn.mixture import GaussianMixture

from tqdm import trange

### BatchDetect module imports

Import project-specific helpers from the `batchdetect` package:

- `load_thal_cross_lot_covs` and related loaders for Thalassemia data.
- `HeavyMixture` and `parametric_bootstrap_lrt` for mixture modeling and
  parametric bootstrap-based likelihood ratio tests.
- Correlation-based clustering utilities:
  `cluster_hierarchical_corr`, `cluster_spectral_corr`,
  `cluster_pca_kmeans_corr`.
- Correlation preprocessing helpers: `normalize_mat` and `get_correlations`.

These functions implement the main batch-detection and clustering methods used
throughout the analysis.


In [2]:
from batchdetect.loader import load_br283_cross_lot_covs
from batchdetect.mixture import HeavyMixture,parametric_bootstrap_lrt

In [3]:
df_thal_likelihoods = pd.read_csv('../likelihoods_br283.csv')
snames = df_thal_likelihoods['Sample Name'].values
likelihoods = df_thal_likelihoods['Likelihood'].values
y_hat = df_thal_likelihoods['Label'].values

### Load  count data and metadata

Load the cross-lot coverage
matrix and associated metadata:

- `counts_thal`: amplicon-level coverage counts per sample.
- `y_thal`: sample labels from the loader (if provided).
- `sample_id`: sample identifiers.
- `features`: amplicon/feature annotations.

These raw counts are used for correlation-based clustering and neighborhood
analysis.


In [4]:
counts_thal,y_thal,sample_id,features,_ = load_br283_cross_lot_covs()
counts_thal = counts_thal[:-1]
y_thal = y_thal[:-1]
sample_id = sample_id[:-1]

In [5]:
index_homozygous_del = [12,13,14]
counts_thal_new = np.delete(counts_thal, index_homozygous_del, axis=0)
likelihoods_new = np.delete(likelihoods, index_homozygous_del)
y_new = np.delete(y_hat, index_homozygous_del)
snames_new = np.delete(snames,index_homozygous_del)

In [6]:
def get_results(dist):
    def null_factory():
        return HeavyMixture(
                n_components=1,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   

    def alt_factory():
        return HeavyMixture(
                n_components=2,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   
    res = parametric_bootstrap_lrt(
            likelihoods_new,  
            null_model_factory=null_factory,
            alt_model_factory=alt_factory,
            n_bootstrap=500,
            random_state=2021,
        )
    return res

In [7]:
res_gaussian = get_results('gaussian')
res_laplace = get_results('laplace')
res_student_t = get_results('student_t')
res_hypsecant = get_results('hypsecant')
res_gennorm = get_results('gennorm')


In [9]:
print("Gaussian p-value: %0.3f"%res_gaussian['p_value'])
print("Laplace p-value: %0.3f"%res_laplace['p_value'])
print("Student-T p-value: %0.3f"%res_student_t['p_value'])
print("Hyp p-value: %0.3f"%res_hypsecant['p_value'])
print("Gennorm p-value: %0.3f"%res_gennorm['p_value'])

Gaussian p-value: 0.086
Laplace p-value: 0.036
Student-T p-value: 0.080
Hyp p-value: 0.786
Gennorm p-value: 0.056
